In [1]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("")

In [3]:
urls = ['https://www.victoriaonmove.com.au/local-removalists.html','https://victoriaonmove.com.au/index.html','https://victoriaonmove.com.au/contact.html']
loader = UnstructuredURLLoader(urls=urls)
data = loader.load()
print(type(data[0]))
print(len(data))

<class 'langchain_core.documents.base.Document'>
3


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=200,
    #length_function=len,
    #is_separator_regex=False,
)

docs = text_splitter.split_documents(data)
print("Total number of documents: ",len(docs))

Total number of documents:  8


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", 
api_key="")

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings)

In [6]:
vector_store.add_documents(documents=docs)

['685967e8-9443-4252-bb8f-5c0e36cdd4a9',
 '97403be3-8fe1-4eb4-b454-7a2a3935aca0',
 'b22b61b3-43be-4e00-a9e2-843e1e08e6f0',
 '8a4f1713-fa4a-4236-947a-8cc345e7edc2',
 '72a9146d-4bba-4ef9-aecb-7433b13094aa',
 'd41ae529-090c-4a5d-9010-cc32671236bc',
 '1e8694d8-ac48-456a-a37e-e0940c9476c5',
 '56428a8b-9cfd-4173-b1be-32ae79e4a6f5']

In [7]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [8]:
retrieved_docs = retriever.invoke("What kind of services they provide?")
len(retrieved_docs)

3

In [9]:
from IPython.display import display, Markdown, HTML

display(Markdown(retrieved_docs[0].page_content))


Icon

Apartment Moving

Efficient and careful relocation services tailored for apartments of all sizes, ensuring smooth transitions to your new home.

Icon

Villa Moving

Comprehensive moving solutions for large residences and villas, handling valuable possessions with utmost care and precision.

Icon

Household Moving

Full-service moving options for households, including packing, loading, transportation, and unpacking services to simplify your move.

Icon

Office Moving

Specialized expertise in office relocations, minimizing downtime and ensuring your business operations continue seamlessly.

Icon

Furniture Moving

Experienced in handling furniture of all sizes and types, ensuring safe transport and setup in your new location.

Icon

Packing and Unpacking Services

Optional packing and unpacking services available to save you time and effort, using high-quality packing materials.

Icon

Customized Moving Plans

In [ ]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=""
)


In [15]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [16]:
# 1. Helper function to format the retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
# 2. Build the LCEL Chain
# This is the "Modern Way": a transparent pipeline using the pipe operator
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 3. Execution
question = "What moving services does Victoria on Move provide?"
response = rag_chain.invoke(question)

print(response)

Victoria on Move offers a range of relocation services, including local and interstate moves, apartment moving, and specialized furniture removals. They also provide optional packing and unpacking services, customized moving plans, and offer transit and public liability insurance. Additionally, they cater to moves for various home sizes, from 1-bedroom apartments to larger homes.
